# Paper 9: WSD-BERT + BiLSTM + Attention + GCN with Neural Seq2Seq Commentary (Nadeem et al., 2025)


In [ ]:
import os
import re
import math
import json
import time
import shutil
import random
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput
from sklearn.metrics import f1_score, accuracy_score

import nltk
for pkg in ["wordnet", "omw-1.4", "averaged_perceptron_tagger", "stopwords"]:
    try:
        nltk.data.find(f"corpora/{pkg}" if pkg != "averaged_perceptron_tagger" else f"taggers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Device: {DEVICE} | GPUs: {NUM_GPUS}")


## 1. Cấu hình Siêu tham số theo Table 6 (Trang 24) của Paper 9


In [ ]:
class Paper9Config:
    PARQUET_CANDIDATES = [
        "/kaggle/input/dap_dts/hotel_reviews.parquet",
        "/kaggle/input/dap-dts/hotel_reviews.parquet",
        "/kaggle/input/hotel-reviews-cram-absa/hotel_reviews.parquet",
        "/kaggle/input/tripadvisor-hotel-reviews/hotel_reviews.parquet",
        "data/df/hotel_reviews.parquet",
        "hotel_reviews.parquet"
    ]
    OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "model")
    LOCAL_MODEL_DIR = Path("model")
    MODEL_SAVE_PATH = OUTPUT_DIR / "modelmethod9.pt"
    LOCAL_MODEL_SAVE_PATH = LOCAL_MODEL_DIR / "modelmethod9.pt"
    HISTORY_SAVE_PATH = OUTPUT_DIR / "history_method9.json"

    BACKBONE_NAME = "bert-base-uncased"
    GEN_MODEL_NAME = "google/flan-t5-small"
    BERT_DIM = 768
    GCN_LAYERS = 2
    GCN_HIDDEN_DIM = 128
    BILSTM_UNITS = 128
    DROPOUT_RATE = 0.5
    NUM_SOFT_PROMPT_TOKENS = 4
    MAX_COMMENT_LEN = 48

    CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
    NUM_ASPECTS = 6
    NUM_SENTIMENT_CLASSES = 4
    NUM_STARS = 5
    MAX_SEQ_LEN = 256

    BATCH_SIZE = 32
    EPOCHS = 6
    LR_BERT = 3e-5
    LR_HEADS = 1e-3
    NUM_WORKERS = 2 if os.name != "nt" else 0
    PIN_MEMORY = torch.cuda.is_available()

cfg = Paper9Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 2. Module Giải nhập nhằng Nghĩa từ WordNet WSD bằng Cosine Similarity (Algorithm 1 & Eq. 2–3 Paper 9)


In [ ]:
ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(cfg.CANONICAL_ASPECTS)}
SENTIMENT_TO_ID = {"Negative": 1, "Neutral": 2, "Positive": 3}
ID_TO_SENTIMENT = {0: "Not Mentioned", 1: "Negative", 2: "Neutral", 3: "Positive"}

HEDGING_MODIFIERS = {
    "moderate", "okay", "ok", "decent", "fair", "acceptable", "average", "slightly",
    "bit", "ordinary", "normal", "passable", "tolerable", "adequate", "fine", "basic"
}

try:
    from nltk.corpus import wordnet as wn
    WORDNET_AVAILABLE = True
except Exception:
    WORDNET_AVAILABLE = False

AMBIGUOUS_HOTEL_TERMS = {
    "sharp", "cold", "warm", "hard", "soft", "small", "high", "low",
    "light", "heavy", "old", "dark", "basic", "quiet", "plain", "rich", "fair", "fine"
}

def enrich_text_with_wordnet_wsd(text: str) -> str:
    raw = unicodedata.normalize("NFC", str(text).strip())
    if not WORDNET_AVAILABLE:
        return raw
    words = re.findall(r"[a-zA-Z]+", raw)
    context_set = {w.lower() for w in words if len(w) > 2}
    sense_glosses = []
    for w in words:
        wl = w.lower()
        if wl in AMBIGUOUS_HOTEL_TERMS:
            synsets = wn.synsets(wl)
            if synsets:
                best_syn = synsets[0]
                best_sim = -1.0
                for syn in synsets[:4]:
                    gloss_words = set(re.findall(r"[a-z]+", (syn.definition() + " " + " ".join(syn.examples())).lower()))
                    if gloss_words:
                        inter = len(context_set & gloss_words)
                        cos_sim = inter / math.sqrt(max(len(context_set), 1) * len(gloss_words))
                        if cos_sim > best_sim:
                            best_sim = cos_sim
                            best_syn = syn
                sense_glosses.append(f"{wl}: {best_syn.definition()}")
                if len(sense_glosses) >= 2:
                    break
    if sense_glosses:
        return raw + " [SEP] " + " ; ".join(sense_glosses)
    return raw

def build_normalized_dependency_adj(attention_mask: torch.Tensor, window: int = 2) -> torch.Tensor:
    bsz, seq_len = attention_mask.size()
    idx = torch.arange(seq_len, device=attention_mask.device)
    dist = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs()
    local_adj = (dist <= window).float().unsqueeze(0).expand(bsz, -1, -1)
    valid_2d = attention_mask.unsqueeze(1).float() * attention_mask.unsqueeze(2).float()
    adj = local_adj * valid_2d
    adj[:, 0, :] = attention_mask.float()
    adj[:, :, 0] = attention_mask.float()
    deg = adj.sum(dim=-1, keepdim=True).clamp(min=1.0)
    inv_sqrt_deg = deg.pow(-0.5)
    norm_adj = inv_sqrt_deg * adj * inv_sqrt_deg.transpose(1, 2)
    return norm_adj

def parse_annotations_safe(anns: Any) -> List[Dict[str, Any]]:
    if anns is None:
        return []
    if isinstance(anns, np.ndarray):
        anns = anns.tolist()
    elif isinstance(anns, str):
        try:
            anns = json.loads(anns)
        except Exception:
            return []
    if not isinstance(anns, list):
        return []
    return [dict(a) for a in anns if isinstance(a, dict)]


## 3. Nạp Tập dữ liệu và Khởi tạo DataLoader


In [ ]:
parquet_path = next((Path(p) for p in cfg.PARQUET_CANDIDATES if Path(p).exists()), None)
if parquet_path is None and Path("/kaggle/input").exists():
    found = list(Path("/kaggle/input").rglob("hotel_reviews.parquet"))
    if found:
        parquet_path = found[0]
if parquet_path is None:
    raise FileNotFoundError("Không tìm thấy hotel_reviews.parquet!")

df_master = pd.read_parquet(parquet_path)
if "ordinal_rating" not in df_master.columns:
    df_master["ordinal_rating"] = df_master["rating_5"].clip(1, 5).astype(int) - 1

df_train = df_master[df_master["split"] == "train"].reset_index(drop=True)
df_val = df_master[df_master["split"] == "val"].reset_index(drop=True)
df_test = df_master[df_master["split"] == "test"].reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)
gen_tokenizer = AutoTokenizer.from_pretrained(cfg.GEN_MODEL_NAME)

class Paper9WSDDataset(Dataset):
    def __init__(self, df: pd.DataFrame, bert_tok, seq2seq_tok, max_len: int = 256, max_comment_len: int = 48):
        self.df = df.reset_index(drop=True)
        self.bert_tok = bert_tok
        self.seq2seq_tok = seq2seq_tok
        self.max_len = max_len
        self.max_comment_len = max_comment_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        raw_text = unicodedata.normalize("NFC", str(row["review_text"]))
        wsd_text = enrich_text_with_wordnet_wsd(raw_text)
        enc = self.bert_tok(
            wsd_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        offsets = enc["offset_mapping"].squeeze(0).numpy()

        ordinal_rating = int(row["ordinal_rating"])
        aspect_sent_labels = torch.zeros(cfg.NUM_ASPECTS, dtype=torch.long)
        aspect_star_labels = torch.full((cfg.NUM_ASPECTS,), ordinal_rating, dtype=torch.long)
        rationale_masks = torch.zeros((cfg.NUM_ASPECTS, self.max_len), dtype=torch.float32)
        gen_span_ids = torch.zeros((cfg.NUM_ASPECTS, self.max_comment_len), dtype=torch.long)
        gen_span_mask = torch.zeros((cfg.NUM_ASPECTS, self.max_comment_len), dtype=torch.long)

        anns = parse_annotations_safe(row.get("aspect_annotations", []))
        votes = defaultdict(list)
        span_texts_map = defaultdict(list)
        for ann in anns:
            asp = ann.get("aspect")
            if asp in ASPECT_ID_MAP:
                a_idx = ASPECT_ID_MAP[asp] - 1
                votes[a_idx].append(SENTIMENT_TO_ID.get(ann.get("sentiment", "Neutral"), 2))
                raw_sp = str(ann.get("span_text", "")).strip()
                if raw_sp:
                    span_texts_map[a_idx].append(raw_sp)
                s_char = int(ann.get("start_char", 0))
                e_char = int(ann.get("end_char", 0))
                for t_i, (ts, te) in enumerate(offsets):
                    if te > ts and ts < e_char and te > s_char:
                        rationale_masks[a_idx, t_i] = 1.0

        for a_idx, v_list in votes.items():
            maj = Counter(v_list).most_common(1)[0][0]
            aspect_sent_labels[a_idx] = maj
            joined_sp = " ".join(span_texts_map[a_idx]).strip()
            sp_words = re.findall(r"[a-z]+", (joined_sp if joined_sp else raw_text).lower())
            has_hedge = any(w in HEDGING_MODIFIERS for w in sp_words)
            if maj == 3:
                aspect_star_labels[a_idx] = 3 if has_hedge else max(3, ordinal_rating)
            elif maj == 2:
                aspect_star_labels[a_idx] = 2
            else:
                aspect_star_labels[a_idx] = 1 if has_hedge else min(1, ordinal_rating)

            if joined_sp:
                t_enc = self.seq2seq_tok(
                    joined_sp,
                    max_length=self.max_comment_len,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                gen_span_ids[a_idx] = t_enc["input_ids"].squeeze(0)
                gen_span_mask[a_idx] = t_enc["attention_mask"].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "aspect_sent_labels": aspect_sent_labels,
            "aspect_star_labels": aspect_star_labels,
            "rationale_masks": rationale_masks,
            "gen_span_ids": gen_span_ids,
            "gen_span_mask": gen_span_mask,
            "ordinal_rating": torch.tensor(ordinal_rating, dtype=torch.long)
        }

train_loader = DataLoader(Paper9WSDDataset(df_train, tokenizer, gen_tokenizer, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN), batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)
val_loader = DataLoader(Paper9WSDDataset(df_val, tokenizer, gen_tokenizer, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN), batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)
test_loader = DataLoader(Paper9WSDDataset(df_test, tokenizer, gen_tokenizer, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN), batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)


## 4. Kiến trúc WSD-BERT + BiLSTM + Attention + 2-Layer GCN Tích hợp Neural Seq2Seq Generator


In [ ]:
class GraphConvLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, norm_adj: torch.Tensor) -> torch.Tensor:
        support = self.linear(x)
        out = torch.bmm(norm_adj, support)
        return self.dropout(F.relu(out))

class Paper9WSD_BiLSTM_GCN(nn.Module):
    def __init__(self, cfg: Paper9Config):
        super().__init__()
        self.num_aspects = cfg.NUM_ASPECTS
        self.max_comment_len = getattr(cfg, "MAX_COMMENT_LEN", 48)
        self.num_prompt_tokens = getattr(cfg, "NUM_SOFT_PROMPT_TOKENS", 4)

        self.bert = AutoModel.from_pretrained(cfg.BACKBONE_NAME)
        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(cfg.GEN_MODEL_NAME)
        self.gen_d_model = self.gen_model.config.d_model
        for param in self.gen_model.parameters():
            param.requires_grad = False

        self.bilstm = nn.LSTM(
            input_size=cfg.BERT_DIM,
            hidden_size=cfg.BILSTM_UNITS,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.bilstm_dropout = nn.Dropout(cfg.DROPOUT_RATE)
        self.attn_queries = nn.Parameter(torch.randn(cfg.NUM_ASPECTS, cfg.BILSTM_UNITS * 2) * 0.02)

        self.gcn1 = GraphConvLayer(cfg.BERT_DIM, cfg.GCN_HIDDEN_DIM, cfg.DROPOUT_RATE)
        self.gcn2 = GraphConvLayer(cfg.GCN_HIDDEN_DIM, cfg.GCN_HIDDEN_DIM, cfg.DROPOUT_RATE)

        combined_dim = cfg.BILSTM_UNITS * 2 + cfg.GCN_HIDDEN_DIM
        self.fusion_gate = nn.Parameter(torch.tensor(0.5))

        self.aspect_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Dropout(cfg.DROPOUT_RATE),
                nn.Linear(combined_dim, cfg.NUM_SENTIMENT_CLASSES)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.aspect_star_heads = nn.ModuleList([
            nn.Sequential(
                nn.Dropout(cfg.DROPOUT_RATE),
                nn.Linear(combined_dim, cfg.NUM_STARS)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.review_star_head = nn.Sequential(
            nn.Dropout(cfg.DROPOUT_RATE),
            nn.Linear(combined_dim, cfg.NUM_STARS)
        )
        self.aspect_soft_prompt = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.GELU(),
            nn.Linear(256, self.num_prompt_tokens * self.gen_d_model),
            nn.LayerNorm(self.num_prompt_tokens * self.gen_d_model)
        )

    def forward(self, input_ids, attention_mask, gen_span_ids=None, gen_span_mask=None):
        bsz = input_ids.size(0)
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_emb = bert_out.last_hidden_state

        lstm_out, _ = self.bilstm(seq_emb)
        lstm_out = self.bilstm_dropout(lstm_out)

        norm_adj = build_normalized_dependency_adj(attention_mask, window=2)
        gcn_h = self.gcn1(seq_emb, norm_adj)
        gcn_h = self.gcn2(gcn_h, norm_adj)

        pad_mask = (attention_mask == 0).unsqueeze(1)
        attn_scores = torch.einsum("bld,ad->bal", lstm_out, self.attn_queries) / math.sqrt(lstm_out.size(-1))
        attn_scores = attn_scores.masked_fill(pad_mask, -1e4)
        attn_weights = F.softmax(attn_scores, dim=-1)
        rationale_probs = torch.sigmoid(attn_scores) * attention_mask.unsqueeze(1).float()

        lstm_Side = torch.einsum("bal,bld->bad", attn_weights, lstm_out)
        gcn_Side = torch.einsum("bal,bld->bad", attn_weights, gcn_h)

        w = torch.sigmoid(self.fusion_gate)
        combined = torch.cat([w * lstm_Side, (1.0 - w) * gcn_Side], dim=-1)

        asp_logits = torch.stack([self.aspect_classifiers[a](combined[:, a, :]) for a in range(self.num_aspects)], dim=1)
        asp_star_logits = torch.stack([self.aspect_star_heads[a](combined[:, a, :]) for a in range(self.num_aspects)], dim=1)
        rev_star_logits = self.review_star_head(combined.mean(dim=1))
        soft_prompts = torch.stack([
            self.aspect_soft_prompt(combined[:, a, :]).view(bsz, self.num_prompt_tokens, self.gen_d_model)
            for a in range(self.num_aspects)
        ], dim=1)

        gen_loss = torch.tensor(0.0, device=input_ids.device)
        if gen_span_ids is not None and gen_span_mask is not None:
            flat_prompts = soft_prompts.view(bsz * self.num_aspects, self.num_prompt_tokens, self.gen_d_model)
            flat_ids = gen_span_ids.view(bsz * self.num_aspects, -1)
            flat_mask = gen_span_mask.view(bsz * self.num_aspects, -1)
            valid_rows = flat_mask.sum(dim=-1) > 1
            if valid_rows.any():
                active_prompts = flat_prompts[valid_rows]
                active_ids = flat_ids[valid_rows]
                active_mask = flat_mask[valid_rows]
                with torch.no_grad():
                    t5_enc = self.gen_model.encoder(input_ids=active_ids, attention_mask=active_mask).last_hidden_state
                    mask_f = active_mask.unsqueeze(-1).float()
                    target_vec = (t5_enc * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
                pred_vec = active_prompts.mean(dim=1)
                gen_loss = (1.0 - F.cosine_similarity(pred_vec, target_vec, dim=-1).mean()) + F.smooth_l1_loss(pred_vec, target_vec)

        return {
            "aspect_logits": asp_logits,
            "aspect_star_logits": asp_star_logits,
            "rev_star_logits": rev_star_logits,
            "rationale_probs": rationale_probs,
            "soft_prompts": soft_prompts,
            "gen_loss": gen_loss
        }

    @torch.no_grad()
    def generate_aspect_commentary(
        self,
        aspect_idx: int,
        aspect_name: str,
        sentiment_label: str,
        star_val: int,
        review_text: str,
        outputs: Dict[str, torch.Tensor],
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        bert_tokenizer,
        seq2seq_tokenizer
    ) -> str:
        device = input_ids.device
        rat_prob = outputs["rationale_probs"][0, aspect_idx]
        valid_len = int(attention_mask[0].sum().item())
        tok_ids = input_ids[0, :valid_len].cpu().tolist()
        scores = rat_prob[:valid_len].cpu().numpy()

        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", review_text.strip()) if s.strip()]
        if len(sentences) > 1:
            scored_sents = []
            for s in sentences:
                s_ids = set(bert_tokenizer.encode(s, add_special_tokens=False))
                hit_scores = [scores[i] for i in range(1, valid_len - 1) if tok_ids[i] in s_ids]
                mean_sc = float(np.mean(hit_scores)) if hit_scores else 0.0
                scored_sents.append((mean_sc, s))
            scored_sents.sort(key=lambda x: x[0], reverse=True)
            focused_evidence = " ".join([s for _, s in scored_sents[:2]])
        else:
            focused_evidence = review_text.strip()

        prompt_str = (
            f"Paraphrase and summarize the guest's opinion on the hotel {aspect_name} "
            f"({sentiment_label}, {star_val}/5 stars) into a short third-person analytical comment: {focused_evidence}"
        )
        p_enc = seq2seq_tokenizer(prompt_str, max_length=192, truncation=True, return_tensors="pt").to(device)
        enc_out = self.gen_model.encoder(input_ids=p_enc["input_ids"], attention_mask=p_enc["attention_mask"])
        soft_p = outputs["soft_prompts"][0:1, aspect_idx, :, :]
        combined_hidden = torch.cat([0.20 * soft_p, enc_out.last_hidden_state], dim=1)
        prompt_mask = torch.ones((1, soft_p.size(1)), dtype=p_enc["attention_mask"].dtype, device=device)
        combined_mask = torch.cat([prompt_mask, p_enc["attention_mask"]], dim=1)

        gen_ids = self.gen_model.generate(
            encoder_outputs=BaseModelOutput(last_hidden_state=combined_hidden),
            attention_mask=combined_mask,
            max_new_tokens=self.max_comment_len,
            min_length=8,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.25,
            early_stopping=True
        )
        return seq2seq_tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()


## 5. Huấn luyện và Kiểm định Mô hình Paper 9


In [ ]:
base_model = Paper9WSD_BiLSTM_GCN(cfg).to(DEVICE)
model = nn.DataParallel(base_model) if NUM_GPUS > 1 else base_model

raw_mod = model.module if isinstance(model, nn.DataParallel) else model
optimizer = torch.optim.AdamW([
    {"params": [p for p in raw_mod.bert.parameters() if p.requires_grad], "lr": cfg.LR_BERT, "weight_decay": 0.01},
    {"params": [p for n, p in raw_mod.named_parameters() if not n.startswith("bert.") and not n.startswith("gen_model.") and p.requires_grad], "lr": cfg.LR_HEADS, "weight_decay": 0.01}
])
ce_loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate_paper9(model, loader):
    model.eval()
    total_loss = 0.0
    star_preds, star_trues = [], []
    asp_preds, asp_trues = [], []
    rat_inter, rat_pred_sum, rat_true_sum = 0.0, 0.0, 0.0
    overlap_rates = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        asp_y = batch["aspect_sent_labels"].to(DEVICE)
        asp_star_y = batch["aspect_star_labels"].to(DEVICE)
        rat_mask = batch["rationale_masks"].to(DEVICE)
        gen_ids = batch["gen_span_ids"].to(DEVICE)
        gen_mask = batch["gen_span_mask"].to(DEVICE)
        star_y = batch["ordinal_rating"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, gen_span_ids=gen_ids, gen_span_mask=gen_mask)
        g_loss = outputs["gen_loss"].mean() if outputs["gen_loss"].dim() > 0 else outputs["gen_loss"]
        loss = (
            ce_loss_fn(outputs["aspect_logits"].view(-1, cfg.NUM_SENTIMENT_CLASSES), asp_y.view(-1))
            + ce_loss_fn(outputs["aspect_star_logits"].view(-1, cfg.NUM_STARS), asp_star_y.view(-1))
            + ce_loss_fn(outputs["rev_star_logits"], star_y)
            + 0.25 * F.binary_cross_entropy(outputs["rationale_probs"].clamp(1e-6, 1.0 - 1e-6), rat_mask)
            + 0.35 * g_loss
        )
        total_loss += loss.item()

        star_preds.extend(outputs["rev_star_logits"].argmax(dim=-1).cpu().numpy())
        star_trues.extend(star_y.cpu().numpy())
        asp_preds.append(outputs["aspect_logits"].argmax(dim=-1).cpu().numpy())
        asp_trues.append(asp_y.cpu().numpy())

        pred_mask = (outputs["rationale_probs"] >= 0.5).float()
        rat_inter += (pred_mask * rat_mask).sum().item()
        rat_pred_sum += pred_mask.sum().item()
        rat_true_sum += rat_mask.sum().item()

        active_per_tok = pred_mask.sum(dim=1)
        leaked = (active_per_tok > 1.0).float().sum().item()
        active = (active_per_tok >= 1.0).float().sum().item()
        overlap_rates.append(leaked / max(active, 1.0))

    y_true_s = np.array(star_trues)
    y_pred_s = np.array(star_preds)
    acc_at_0 = accuracy_score(y_true_s, y_pred_s) * 100.0
    per_star_mae = [np.abs(y_true_s[y_true_s == k] - y_pred_s[y_true_s == k]).mean() for k in range(5) if np.any(y_true_s == k)]
    macro_mae = float(np.mean(per_star_mae)) if per_star_mae else 0.0

    Y_true_a = np.vstack(asp_trues)
    Y_pred_a = np.vstack(asp_preds)
    asp_f1s = [f1_score(Y_true_a[:, a], Y_pred_a[:, a], average="macro", zero_division=0) * 100.0 for a in range(cfg.NUM_ASPECTS)]
    macro_asp_f1 = float(np.mean(asp_f1s))
    rationale_f1 = (2.0 * rat_inter / max(rat_pred_sum + rat_true_sum, 1.0)) * 100.0

    return {
        "val_loss": round(total_loss / max(len(loader), 1), 4),
        "acc_at_0": round(acc_at_0, 2),
        "aspect_macro_f1": round(macro_asp_f1, 2),
        "macro_mae": round(macro_mae, 4),
        "rationale_f1": round(rationale_f1, 2),
        "leakage_rate": round(float(np.mean(overlap_rates)) * 100.0, 2),
        "per_aspect_f1": dict(zip(cfg.CANONICAL_ASPECTS, [round(x, 2) for x in asp_f1s]))
    }

history = []
best_f1 = -1.0

for epoch in range(1, cfg.EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        asp_y = batch["aspect_sent_labels"].to(DEVICE)
        asp_star_y = batch["aspect_star_labels"].to(DEVICE)
        rat_mask = batch["rationale_masks"].to(DEVICE)
        gen_ids = batch["gen_span_ids"].to(DEVICE)
        gen_mask = batch["gen_span_mask"].to(DEVICE)
        star_y = batch["ordinal_rating"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, gen_span_ids=gen_ids, gen_span_mask=gen_mask)
        g_loss = outputs["gen_loss"].mean() if outputs["gen_loss"].dim() > 0 else outputs["gen_loss"]
        loss = (
            ce_loss_fn(outputs["aspect_logits"].view(-1, cfg.NUM_SENTIMENT_CLASSES), asp_y.view(-1))
            + ce_loss_fn(outputs["aspect_star_logits"].view(-1, cfg.NUM_STARS), asp_star_y.view(-1))
            + ce_loss_fn(outputs["rev_star_logits"], star_y)
            + 0.25 * F.binary_cross_entropy(outputs["rationale_probs"].clamp(1e-6, 1.0 - 1e-6), rat_mask)
            + 0.35 * g_loss
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()

    val_metrics = evaluate_paper9(model, val_loader)
    if val_metrics["aspect_macro_f1"] > best_f1:
        best_f1 = val_metrics["aspect_macro_f1"]
        raw_state = (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
        torch.save({
            "epoch": epoch,
            "method": "Paper 9: WSD-BERT + BiLSTM + Attention + GCN (Nadeem et al., 2025)",
            "architecture": "paper9_wsd_bilstm_gcn",
            "gen_model_name": cfg.GEN_MODEL_NAME,
            "model_state_dict": raw_state,
            "canonical_aspects": cfg.CANONICAL_ASPECTS,
            "val_metrics": val_metrics
        }, cfg.MODEL_SAVE_PATH)
        if cfg.LOCAL_MODEL_SAVE_PATH.resolve() != cfg.MODEL_SAVE_PATH.resolve():
            shutil.copy2(cfg.MODEL_SAVE_PATH, cfg.LOCAL_MODEL_SAVE_PATH)

    rec = {"epoch": epoch, "train_loss": round(running_loss / max(len(train_loader), 1), 4), **val_metrics, "time_s": round(time.time() - t0, 1)}
    history.append(rec)
    print(f"Epoch {epoch}/{cfg.EPOCHS} | Train Loss: {rec['train_loss']:.4f} | Val Acc@0: {val_metrics['acc_at_0']:.2f}% | 6-Aspect F1: {val_metrics['aspect_macro_f1']:.2f}% | Macro-MAE: {val_metrics['macro_mae']:.4f} | Rat-F1: {val_metrics['rationale_f1']:.2f}%")

test_metrics = evaluate_paper9(model, test_loader)
print("Test Metrics (Paper 9):", test_metrics)
with open(cfg.HISTORY_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump({"history": history, "test_metrics": test_metrics}, f, indent=2)


## 6. Suy luận trên Review Mới 100% Bằng Trọng số Neural


In [ ]:
@torch.no_grad()
def predict_new_review_paper9(new_review_text: str, model: nn.Module, bert_tokenizer, seq2seq_tokenizer) -> str:
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.eval()
    raw_text = unicodedata.normalize("NFC", str(new_review_text).strip())
    wsd_text = enrich_text_with_wordnet_wsd(raw_text)
    enc = bert_tokenizer(wsd_text, max_length=cfg.MAX_SEQ_LEN, padding="max_length", truncation=True, return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    outputs = raw_model(input_ids=input_ids, attention_mask=attention_mask)
    asp_preds = outputs["aspect_logits"].squeeze(0).argmax(dim=-1).cpu().numpy()
    asp_stars = outputs["aspect_star_logits"].squeeze(0).argmax(dim=-1).cpu().numpy() + 1

    mentioned = []
    per_asp_str = {}
    for a_idx, asp in enumerate(cfg.CANONICAL_ASPECTS):
        cls_id = int(asp_preds[a_idx])
        if cls_id != 0:
            mentioned.append(asp)
            star_v = int(asp_stars[a_idx])
            sent_lbl = ID_TO_SENTIMENT[cls_id]
            gen_comment = raw_model.generate_aspect_commentary(
                aspect_idx=a_idx,
                aspect_name=asp,
                sentiment_label=sent_lbl,
                star_val=star_v,
                review_text=raw_text,
                outputs=outputs,
                input_ids=input_ids,
                attention_mask=attention_mask,
                bert_tokenizer=bert_tokenizer,
                seq2seq_tokenizer=seq2seq_tokenizer
            )
            per_asp_str[asp] = f"{star_v}/5 ★ ({sent_lbl}) - {gen_comment}"
        else:
            per_asp_str[asp] = ""

    lines = [f"- Aspect: {', '.join(mentioned) if mentioned else 'None'}"]
    for asp in cfg.CANONICAL_ASPECTS:
        lines.append(f"- {asp}: {per_asp_str[asp]}" if per_asp_str[asp] else f"- {asp}:")
    return "\n".join(lines)
